# DL vertex input image generation

This notebook is designed to take CSV files generated by the <code>PrepareTrainingSample</code> function of the <code>DlVertexingAlgorithm</code>. This algorithm generates CSV files for each of the U. V and W views and the code below will run over each of those files.
    
Most of the cells below will not need any editing, but at the very bottom of the notebook you will find some additional markdown that describes what you may need to edit (essentially just some file locations).

In [ ]:
# Automatically reload external libraries that change
%reload_ext autoreload
%autoreload 2

# If a matplotlib plot command is issued, display the results in the notebook
%matplotlib inline

In [ ]:
import cv2
import csv
import numpy as np
import os
import glob
from tqdm.notebook import tqdm

import h5py
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
wire_pitch = { "U": 0.46669998765, "V": 0.46669998765, "W": 0.479000002146 }
wire_pitch = { "UV": 0.3, "W": 0.3, "U": 0.3, "V": 0.3 }
# drift_step = 0.5
drift_step = 0.3

# thresholds must be defined/imported wherever make_input_histogram is used;
# left as a module-level global to match the original code's expectations.
thresholds = None

def make_input_histogram(x, z, adc, vertex, x_bounds, z_bounds, image_size, view):
    global thresholds
    image_height, image_width = image_size
    x_min, x_max = x_bounds
    z_min, z_max = z_bounds
    
    # Update the span if image is too small
    r_span = np.sqrt((x_max - x_min)**2 + (z_max - z_min)**2)
    
    x_bins = np.linspace(x_min - 0.5 * drift_step, x_max + 0.5 * drift_step, image_width + 1)
    z_bins = np.linspace(z_min - 0.5 * wire_pitch[view], z_max + 0.5 * wire_pitch[view], image_height + 1)
    
    phx = np.digitize(x, x_bins) - 1
    phz = np.digitize(z, z_bins) - 1
        
    pvx = np.digitize(vertex[0], x_bins) - 1
    pvz = np.digitize(vertex[1], z_bins) - 1

    input_histogram, _, _ = np.histogram2d(z, x, bins=[z_bins, x_bins], weights=adc)
    input_histogram = input_histogram.astype(float)

    # ATTN: Need to set the dtype here or truth will be of type 8-bit uint before scaling, which will result in
    # incorrect distances for dr > 255
    truth_histogram = np.zeros_like(input_histogram, dtype=float)
    
    # Handle case where vertex is outside of the hit bounding box. Note, it may still be possible for the correction
    # to move the vertex outside the bounding box in the opposite direction (unlikely, but think about it)
    # other way to do this is to bound the image including the vertex location
    if pvx < 0: # underflow
        pvx = -(np.digitize(x_bins[0] + (x_bins[0] - vertex[0]), x_bins) - 1)
    elif pvx >= (len(x_bins) - 1): # overflow
        pvx = np.digitize(x_bins[-1] - (vertex[0] - x_bins[-1]), x_bins) - 1
    if pvz < 0: # underflow
        pvz = -(np.digitize(z_bins[0] + (z_bins[0] - vertex[1]), z_bins) - 1)
    elif pvz >= (len(z_bins) - 1): # overflow
        pvz = np.digitize(z_bins[-1] - (vertex[1] - z_bins[-1]), z_bins) - 1

    dr = np.sqrt((phx - pvx)**2 + (phz - pvz)**2)
    class_histogram = np.zeros_like(truth_histogram)
    for i in range(len(phx)):
        truth_histogram[phz[i], phx[i]] = dr[i]
    truth_min, truth_max = np.min(truth_histogram), np.max(truth_histogram)
    if truth_max > truth_min:
        truth_histogram = (truth_histogram - truth_min) / np.ceil(np.sqrt(2*(image_height - 1)**2))
        for i in range(len(phx)):
            cls = np.digitize(truth_histogram[phz[i], phx[i]], thresholds)
            class_histogram[phz[i], phx[i]] = cls if cls < len(thresholds) else len(thresholds) - 1
    else:
        class_histogram = np.zeros_like(input_histogram)

    return input_histogram, class_histogram.astype('uint8')

In [ ]:
# preprocess.py

def display(hits_x, hits_z, vrt_x, vrt_z):
    """Displays an image of the hits and vertices.

        Args:
            hits_x: list of x coordinates for hits
            hits_z: list of z coordinates for hits
            vrt_x: list of x coordinates for vertices
            vrt_z: list of z coordinates for vertices
    """
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(5, 5))
    plt.scatter(hits_x, hits_z, c='black', s=20, alpha=1.0, label="Hits")
    plt.scatter(vrt_x, vrt_z, c='red', s=50, alpha=1.0, label="Vertices")
    plt.legend()
    plt.show()
    plt.close(fig)


def process_event(data, image_size, view):
    """Generate the training/validation set images for a single event.

        The input data has the format:
        N Vertices,M Hits,N*{vertex x, vertex y},M*{hit x, hit z, adc}

        The first vertex in the list is always the primary vertex

        Args:
            data: The event's fields (already trimmed of leading/trailing sentinel columns)
            image_size: (height, width) of the output histograms
            view: Detector view

        Returns:
            (input_histogram, truth_histogram) tuple, or None if the event has no hits
    """
    nv_coords = 2
    nh_coords = 3
    nuance = int(data.pop(0))
    n_vertices = int(data.pop(0))

    v_start, v_finish = 0, nv_coords * n_vertices
    vx = np.array(data[v_start:v_finish:2], dtype=float)
    vz = np.array(data[v_start + 1:v_finish:2], dtype=float)
    b_start = v_finish
    x_min, x_max = float(data[b_start]), float(data[b_start + 1])
    z_min, z_max = float(data[b_start + 2]), float(data[b_start + 3])

    n_hits = int(data[b_start + 4])
    h_start, h_finish = b_start + 5, b_start + 5 + nh_coords * n_hits
    length = len(data[h_start:])

    if length != (n_hits * nh_coords):
        print('Missing information in input file')
        print(n_hits, length)
        return None

    hx = np.array(data[h_start:h_finish:nh_coords], dtype=float)
    hz = np.array(data[h_start + 1:h_finish:nh_coords], dtype=float)
    hadc = np.array(data[h_start + 2:h_finish:nh_coords], dtype=float)

    if hx.size == 0 or hz.size == 0 or hadc.size == 0:
        return None

    return make_input_histogram(
        hx, hz, hadc, (vx[0], vz[0]),
        (x_min, x_max), (z_min, z_max), image_size, view
    )


def process_file(input_file, output_folder, view, image_size=(128, 128), write_batch_size=500,
                  compression='lzf'):
    """Generate a single HDF5 file of training/validation images for all events in a file.

        The input CSV file has the format:
        Date/Time,N Vertices,M Hits,N*{vertex x, vertex y},M*{hit x, hit y},EOL

        The first vertex in the list is always the primary vertex

        Writes to <output_folder>/data.h5, containing two resizable, chunked datasets:
            'hits'  - shape (N, H, W), float32
            'truth' - shape (N, H, W), int64

        Both datasets use chunks=(1, H, W) - one event per chunk - so that a random-access
        read of a single sample during training only has to decompress that one chunk,
        rather than an entire shard (as the previous .npz-shard format required). Because
        hits and truth are written into the SAME file in the same flush() call, they can
        never end up with mismatched counts the way separate per-array shard files could if
        a write was interrupted partway through (e.g. by a full disk).

        Events are buffered in memory and written to the HDF5 file in batches of
        write_batch_size to amortize the per-call overhead of resizing/writing - this only
        affects write throughput, not the on-disk chunk layout (still one event per chunk),
        so it has no effect on later random-access read performance.

        Args:
            input_file: a CSV file containing event information
            output_folder: The folder to write data.h5 into
            view: Detector view
            image_size: The output image size as a tuple (height, width) (default: (128, 128))
            write_batch_size: Number of events to buffer before each write to the HDF5 file
                (default: 500)
            compression: h5py compression filter for both datasets (default: 'lzf' - low CPU
                overhead, decent ratio, and unlike gzip doesn't need extra native libs at
                read time; pass None to disable compression entirely for maximum read speed
                at the cost of more disk space)
    """
    os.makedirs(output_folder, exist_ok=True)
    h5_path = os.path.join(output_folder, "data.h5")
    height, width = image_size

    with open(input_file, 'r') as f:
        num_events = sum(1 for _ in f)

    hits_buf, truth_buf = [], []

    with h5py.File(h5_path, 'w') as hf, open(input_file, 'r') as f:
        hits_ds = hf.create_dataset(
            'hits', shape=(0, height, width), maxshape=(None, height, width),
            chunks=(1, height, width), dtype='float32', compression=compression,
        )
        truth_ds = hf.create_dataset(
            # uint8 matches what make_input_histogram actually emits (class labels 0 ..
            # len(thresholds)-1). Storing a wider dtype here would bloat every truth chunk
            # for no benefit - data.py upcasts to torch.long on read regardless of what's
            # stored on disk.
            'truth', shape=(0, height, width), maxshape=(None, height, width),
            chunks=(1, height, width), dtype='uint8', compression=compression,
        )

        def flush():
            nonlocal hits_buf, truth_buf
            if not hits_buf:
                return
            n = len(hits_buf)
            old = hits_ds.shape[0]
            hits_ds.resize(old + n, axis=0)
            truth_ds.resize(old + n, axis=0)
            hits_ds[old:old + n] = np.stack(hits_buf)
            truth_ds[old:old + n] = np.stack(truth_buf)
            hits_buf, truth_buf = [], []

        reader = csv.reader(f)
        for row in tqdm(reader, desc=f"Processing view {view}", miniters=100, total=num_events):
            result = process_event(row[1:-1], image_size, view)
            if result is None:
                continue
            input_histogram, truth_histogram = result
            hits_buf.append(input_histogram)
            truth_buf.append(truth_histogram)
            if len(hits_buf) >= write_batch_size:
                flush()

        flush()  # final partial batch
        hf.attrs['n_samples'] = hits_ds.shape[0]
        hf.attrs['view'] = view

# Edit below this point

The details that might change between different contexts are the input and output file locations, the class thresholds and potentially the number of passes.

The input files are specified by the <code>file_prefix</code> variable - the <code>PrepareTrainingSample</code> function automatically tags the files with their respective views, so you should omit the view and file type from the specification.

The output location is specified by <code>global_path</code>, within which <code>Hit</code> and <code>Truth</code> folders will be created to store the input and target output images for training.

The thresholds are specified by the <code>thresholds</code> variable, a global variable referenced by <code>make_input_histograms</code>. It is critical that the values here match those specified in the Pandora XML specification for the vertexing algorithm.

In general, you will want to implement a two pass approach to networking, as this will likely greatly enhance vertex resolution, but if you may only want one pass due to higher pass dependence on earlier passes for CSV generation, you can just alter the <code>vertex_pass</code> loop list to run over the selected pass. There is no explicit dependency between the passes in this notebook.

Once you're happy with these values, you can just run the entire notebook from top to bottom and, after some time, you'll have a set of input/truth images that can be used to train the networks.

In [ ]:
import matplotlib.pyplot as plt

from __future__ import annotations
from cycler import cycler

PlotStyle = {
    'axes.ymargin': 0.1,
    'legend.frameon': False,
    'xaxis.labellocation': 'right',
    'yaxis.labellocation': 'top',
    
    'axes.formatter.limits': (-2, 3),

    # 2. ax.ticklabel_format(useMathText=True)
    'axes.formatter.use_mathtext': True, 

    # 3. ax.minorticks_on()
    'xtick.minor.visible': True, 
    'ytick.minor.visible': True, 

    'xtick.major.size': 6, 
    'ytick.major.size': 6, 

    'xtick.labelsize': 14, 
    'ytick.labelsize': 14, 

    'xtick.direction': 'in', 
    'ytick.direction': 'in', 

    'xtick.top': True, 
    'ytick.right': True, 

    # 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
    'xtick.minor.size': 3, 
    'ytick.minor.size': 3, 

    # Note: Direction, top, and right settings automatically apply to minor ticks 
    # when set globally, but you can explicitly ensure they mirror major ticks.
    'xtick.minor.top': True, 
    'ytick.minor.right': True, 

    'axes.xmargin': 0.0, 

    'legend.title_fontsize': 16, 
    'legend.fontsize': 14, 
    'axes.labelsize': 17, 
    'axes.titlesize': 16,

    'legend.handleheight': 1, 
    'legend.handlelength': 1.2, 

    'savefig.dpi': 300, 

    'axes.prop_cycle': (
        cycler('color', ["#0d49fb", "#e6091c", "#26eb47", "#8936df", "#fec32d", "#25d7fd"]) + 
        cycler('ls', ['-', '--', '-.', ':', '-', '--'])
    ), 
    'savefig.transparent': True,
    'savefig.bbox': 'tight',
}

plt.style.use(PlotStyle)

thresholds = [0., 0.00275, 0.00825, 0.01925, 0.03575, 0.05775, 0.08525, \
              0.12375, 0.15125, 0.20625, 0.26125, 0.31625, 0.37125, 0.42625, \
              0.50875, 0.59125, 0.67375, 0.75625, 0.85, 1.0]

# Pass 1: creating images for the first pass

In [ ]:
for vertex_pass in [1]:
    for beam in tqdm(['BNB', 'NuMI']):
        for view in tqdm(['W', 'UV']):
            for flavour in tqdm(['numu', 'nue']):
                file_prefix = (
                    f'/exp/icarus/data/users/msotgia/vertexStudies/forTraining/pass{vertex_pass}/'
                    f'{beam}/{flavour}/ICARUS_DLVertexTrain_Pass{vertex_pass}_CaloHitList{view}.csv'
                )
                image_size = (256, 256) if vertex_pass == 1 else (192, 192)
        
                global_path = os.path.join(f"/home/msotgia/vertexOnEaf/ICARUS_DlVertex_HDF5/{beam}/{flavour}/Pass{vertex_pass}", f"Images{view}")
                process_file(file_prefix, global_path, view, image_size = image_size)

# Pass 2: creating images for the second vertex pass

In [ ]:
for vertex_pass in [2]:
    for beam in tqdm(['BNB', 'NuMI']):
        for view in tqdm(['W', 'UV']):
            for flavour in tqdm(['numu', 'nue']):
                file_prefix = (
                    f'/exp/icarus/data/users/msotgia/vertexStudies/forTraining/pass{vertex_pass}/'
                    f'{beam}/{flavour}/ICARUS_DLVertexTrain_Pass{vertex_pass}_CaloHitList{view}.csv'
                )
                image_size = (256, 256) if vertex_pass == 1 else (192, 192)
        
                global_path = os.path.join(f"/home/msotgia/vertexOnEaf/ICARUS_DlVertex_HDF5/{beam}/{flavour}/Pass{vertex_pass}", f"Images{view}")
                process_file(file_prefix, global_path, view, image_size = image_size)

In [ ]:
for vertex_pass in [2]:
        for view in tqdm(['W', 'UV']):
            file_prefix = (
                f'/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/'
                f'ICARUS_DLVertexTrain_Pass{vertex_pass}_CaloHitList{view}.csv'
            )
            image_size = (256, 256) if vertex_pass == 1 else (192, 192)
    
            global_path = os.path.join(
                f"/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/", 
                f"Images{view}"
            )
            process_file(file_prefix, global_path, view, image_size = image_size)

# Producing the images 

A couple of helper utilities for producing some plots that might be used for presentation

In [ ]:
from typing import Literal

def plot_event(h5_path, event_idx, plane, out_path=None, what: Literal['truth', 'hits'] = 'truth'):
    with h5py.File(h5_path, 'r') as hf:
        hits = hf['hits'][event_idx]      # shape (H, W), float32
        truth = hf['truth'][event_idx]    # shape (H, W), uint8 class labels
        view = hf.attrs.get('view', plane)

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )
    if what == 'hits':
        ax.imshow(hits, cmap='jet')
        ax.set_title(f'View {plane} / Hits', color='gray', loc='right')

    if what == 'truth':
        mpbl = ax.imshow(truth, cmap='viridis')
        fig.colorbar(mpbl, ax=ax, label='Classes')
        ax.set_title(f'View {plane} / Truth', color='gray', loc='right')

    ax.set(
        ylabel=f'{view} wires $\\longrightarrow$',
        xlabel='Time coordinate $\\longrightarrow$'
    )
    
    if out_path:
        fig.savefig(out_path, bbox_inches='tight')
    return fig, ax


def plot_event_grid(h5_path, event_indices, plane_label=None, out_path=None, 
                    what: Literal['truth', 'hits'] = 'truth', tag=None):
    """Plot hits (top row) and truth classes (bottom row) for several events.

    Args:
        h5_path: path to a data.h5 produced by process_file
        event_indices: list/array of row indices into the 'hits'/'truth' datasets
        plane_label: optional string to show in the title (falls back to hf.attrs['view'])
        out_path: if given, saves the figure here
        cmap: colormap for the hits row
        class_cmap: colormap for the truth row
    """
    with h5py.File(h5_path, 'r') as hf:
        view = plane_label or hf.attrs.get('view', '')
        n_classes = None
        hits = [hf['hits'][i] for i in event_indices]
        truths = [hf['truth'][i] for i in event_indices]

    n_events = len(event_indices)
    fig, axs = plt.subplots(
        figsize=(3.5 * n_events, 4), ncols=n_events, sharey=True
    )

    fig.align_xlabels()
    fig.tight_layout()
    fig.subplots_adjust(wspace=0.0, hspace=0.0)

    # Determine a common color scale for the truth row so classes are comparable
    truth_max = max(t.max() for t in truths) if truths else 1

    for col, (idx, h, t) in enumerate(zip(event_indices, hits, truths)):
        if what=='hits':
            ax_hit = axs[col]
            ax_hit.imshow(h, cmap='jet')
            ax_hit.set(xticks=[], yticks=[])
            # ax_hit.format(title=f'Event {idx}')
            

        if what=='truth':
            ax_truth = axs[col]
            mpbl = ax_truth.imshow(t, cmap='gist_heat_r', vmin=0, vmax=truth_max)
            ax_truth.set(xticks=[], yticks=[])

    if what=='hits':
        axs[0].set(ylabel=f'{view} wires $\\longrightarrow$')
    if what=='truth':
        axs[0].set(ylabel=f'{view} wires $\\longrightarrow$')

    if what=='hits':
        axs[-1].set_title(f'View {plane} / Hits', color='gray', loc='right')

    if what=='truth':
        axs[-1].set_title(f'View {plane} / Truth classes', color='gray', loc='right')

    if tag:
        axs[0].set_title(tag, color='gray', loc='left')

    # fig.colorbar(mpbl, loc='r', label='Classes', frameon=False)
    axs[-1].set(xlabel='Time coordinate $\\longrightarrow$')

    if out_path:
        fig.savefig(out_path, bbox_inches='tight')
    return fig, axs



In [ ]:
# Usage mirroring your original loop, one h5 file per plane/view
for plane in ['W']:
    h5_path = f"/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass2/Images{plane}/data.h5"
    plot_event(
        h5_path, event_idx=1, plane=plane,
        out_path=f'/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass2/PlotVertexClass_{plane}.pdf',
        what='truth'
    )

In [ ]:
# Usage — mirrors your original per-plane loop

what='truth'
vertex_pass=2
for plane in ['W']:
    h5_path = f"/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/Images{plane}/data.h5"
    plot_event_grid(
        h5_path, event_indices=[0, 4, 9, 18],
        out_path=f'/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/PlotVertexClass_{plane}_{what}.pdf',
        what=what, tag=r'ICARUS MC BNB $\nu_\mathrm{e}$'
    )

In [ ]:
what='truth'
vertex_pass=2
for plane in ['UV']:
    h5_path = f"/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/Images{plane}/data.h5"
    plot_event_grid(
        h5_path, event_indices=[0, 4, 9, 18],
        out_path=f'/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass{vertex_pass}/PlotVertexClass_{plane}_{what}.pdf',
        what=what, tag=r'ICARUS MC BNB $\nu_\mathrm{e}$'
    )

In [ ]:
what='truth'
for plane in ['UV']:
    h5_path = f"/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass2/Images{plane}/data.h5"
    plot_event_grid(
        h5_path, event_indices=np.arange(20, 39),
        out_path=f'/exp/icarus/app/users/msotgia/analysis/VertexRepository/vertexUtilities/tmp/testPass2/PlotVertexClass_{plane}_{what}.pdf',
        what=what
    )